# Gaussian Filter

In [ ]:
import sys; sys.path.insert(0, '../'); from lib import *;
figure_features()

OPT  = {
    "MICRO_SEC":   True,                # Time in microseconds (True/False)
    "NORM":        False,               # Runs can be displayed normalised (True/False)
    "ALIGN":       True,                # Aligns waveforms in peaktime (True/False)
    "LOGY":        False,               # Runs can be displayed in logy (True/False)
    "SHOW_AVE":    "",                  # If computed, vis will show average (AveWvf,AveWvfSPE,etc.)
    "SHOW_PARAM":  False,               # Print terminal information (True/False)
    "CHARGE_KEY":  "ChargeAveRange",    # Select charge info to be displayed. Default: "ChargeAveRange" (if computed)
    "PEAK_FINDER": False,               # Finds possible peaks in the window (True/False)
    "LEGEND":      True,                # Shows plot legend (True/False)
    "SHOW":        True
    }

In [ ]:
info=read_input_file('MegaCell_LAr_v2');
PRESET="ALL";
runs=[103];
channels=[0];

my_runs=load_npy(runs,channels,info,preset=PRESET,debug=False) #Struct my_runs[run][channel]['Variable'][event]

In [ ]:
from scipy.ndimage import gaussian_filter

#Select an event
run=103;
c=0;
ev=np.random.choice(np.arange(len(my_runs[run][c]['RawADC'])))

RawData=my_runs[run][c]['RawADC'][ev];
sigma=20;
CleanData=gaussian_filter(RawData,sigma);

Base line and Integration Range

In [ ]:
Ped=np.mean(CleanData[:my_runs[run][c]['RawPedLim']]);
final=np.argmax(CleanData)+np.where(CleanData[np.argmax(CleanData):]<Ped)[0][0];
inicio=np.where(CleanData[:np.argmax(CleanData)]<Ped)[0][-1];

In [ ]:
fig1=go.Figure()
fig1.add_trace(go.Scatter(x=np.arange(len(RawData)),y=RawData,name='Raw Signal'))
fig1.add_trace(go.Scatter(x=np.arange(len(RawData)),y=CleanData,name='Filtered Signal'))
fig1.add_hline(y=Ped)
fig1.add_vline(x=final)
fig1.add_vline(x=inicio)
fig1.update_layout(title='Event Filtered find ranges',title_x=0.5,template='presentation')
fig1.update_xaxes(title='Ticks')
fig1.update_yaxes(title='ADC counts')

In [ ]:
filtered_signals=gaussian_filter(my_runs[run][c]['RawADC'],sigma)

In [ ]:
AnaADC=(my_runs[run][c]['RawADC'].T-np.mean(filtered_signals[:,:my_runs[run][c]['RawPedLim']],axis=1).T).T;

In [ ]:
Maximos=np.argmax(filtered_signals,axis=1)
eve=np.arange(len(AnaADC));
Charge=[];
ped=np.mean(filtered_signals[:,:my_runs[run][c]['RawPedLim']],axis=1)
for i in eve:
    init=np.where(filtered_signals[i,:np.argmax(filtered_signals[i])]<ped[i])[0][-1];
    final=Maximos[i]+np.where(filtered_signals[i,Maximos[i]:]<ped[i])[0][0];#Puede ser que esté vacío (añadir if)
    Charge.append(np.sum(AnaADC[i,init:final]));

In [ ]:
evento=12376;
MeanAnaADC=np.mean(AnaADC,axis=0)
fig2=go.Figure()
fig2.add_trace(go.Scatter(x=np.arange(len(MeanAnaADC)),y=MeanAnaADC))
fig2.add_trace(go.Scatter(x=np.arange(len(MeanAnaADC)),y=AnaADC[evento]))

In [ ]:
print(len(MeanAnaADC))